In [ ]:
import csv
import re
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

URL = "https://en.wikipedia.org/wiki/List_of_Colorado_fourteeners"
BASE = "https://en.wikipedia.org"

HEADERS = {"User-Agent": "co-14ers-scraper/1.0"}
SESSION = requests.Session()

OUT_PATH = "co_14ers.csv"


def dms_to_decimal(dms: str) -> float:
    """
    Convert DMS like '38°50′38″N' or '105°02′42″W' to decimal.
    Also accepts strings with degrees/minutes/seconds symbols in various forms.
    """
    s = dms.strip()

    # Direction
    direction = None
    mdir = re.search(r"([NSEW])\s*$", s)
    if mdir:
        direction = mdir.group(1)
        s = re.sub(r"[NSEW]\s*$", "", s).strip()

    # Extract numbers (deg, min, sec). Some entries may omit seconds.
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?", s)
    if not nums:
        raise ValueError(f"Could not parse DMS: {dms}")

    deg = float(nums[0])
    minutes = float(nums[1]) if len(nums) > 1 else 0.0
    seconds = float(nums[2]) if len(nums) > 2 else 0.0

    dec = abs(deg) + minutes / 60.0 + seconds / 3600.0

    # Sign handling
    if direction in ("S", "W"):
        dec = -dec
    elif direction is None:
        # If no direction, preserve sign of degree if present
        if deg < 0:
            dec = -dec

    return dec


def parse_coord(value: str) -> float:
    """
    Parse a coordinate cell that might be:
      - decimal: '38.84083'
      - DMS: '38°50′27″N'
      - or include extra whitespace/notes
    """
    v = value.strip()

    # Try decimal first (grab first float-looking token)
    m = re.search(r"[-+]?\d+(?:\.\d+)?", v)
    if m:
        candidate = m.group(0)
        # If it looks like a plain decimal (and not degrees-only from DMS), accept
        # Heuristic: if the original contains degree symbol or N/S/E/W, treat as DMS
        if any(sym in v for sym in ("°", "′", "″", "N", "S", "E", "W")):
            return dms_to_decimal(v)
        return float(candidate)

    # Fallback to DMS parser
    return dms_to_decimal(v)


def make_peak_id(lat, lon):
    """
    Round/pad lat/lon to 5 decimals and generate a fixed-length numeric peak_id.
    Returns: (peak_id, norm_lat, norm_lon)
    """
    lat_str = f"{lat:.5f}"
    lon_str = f"{lon:.5f}"
    peak_id = re.sub(r"[^0-9]", "", lat_str + lon_str)
    return peak_id, float(lat_str), float(lon_str)


def clean_peak_name(name: str) -> str:
    """
    Optional cleanup. Keeps it conservative:
    - remove trailing ' (Colorado)' if present
    """
    name = name.strip()
    name = re.sub(r"\s*\(Colorado\)$", "", name)
    return name


def find_tables_with_lat_lon(soup: BeautifulSoup):
    """
    Return wikitable(s) that contain Latitude and Longitude columns.
    """
    tables = soup.find_all("table", class_=lambda c: c and "wikitable" in c)
    good = []

    for t in tables:
        header_row = t.find("tr")
        if not header_row:
            continue
        headers = [th.get_text(" ", strip=True).lower() for th in header_row.find_all(["th", "td"])]
        if any("latitude" in h for h in headers) and any("longitude" in h for h in headers):
            good.append((t, headers))

    return good


def get_col_index(headers, needle: str):
    """
    Find a column index by matching substring in header.
    """
    needle = needle.lower()
    for i, h in enumerate(headers):
        if needle in h:
            return i
    return None


def scrape_colorado_14ers():
    r = SESSION.get(URL, headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    tables = find_tables_with_lat_lon(soup)
    if not tables:
        raise RuntimeError("Couldn't find any wikitable with Latitude/Longitude columns. Page structure may have changed.")

    rows = []
    seen = set()  # avoid dupes if multiple tables overlap

    for table, headers in tables:
        lat_idx = get_col_index(headers, "latitude")
        lon_idx = get_col_index(headers, "longitude")

        # Name column varies; common headers include "mountain", "peak", "name"
        name_idx = (
            get_col_index(headers, "mountain")
            or get_col_index(headers, "peak")
            or get_col_index(headers, "name")
        )
        if name_idx is None or lat_idx is None or lon_idx is None:
            continue

        for tr in table.find_all("tr")[1:]:
            tds = tr.find_all(["td", "th"])
            if len(tds) <= max(name_idx, lat_idx, lon_idx):
                continue

            name_cell = tds[name_idx]
            # Prefer linked article text if present
            a = name_cell.find("a", href=True)
            peak_name = a.get_text(" ", strip=True) if a else name_cell.get_text(" ", strip=True)
            peak_name = clean_peak_name(peak_name)

            lat_raw = tds[lat_idx].get_text(" ", strip=True)
            lon_raw = tds[lon_idx].get_text(" ", strip=True)

            try:
                lat = parse_coord(lat_raw)
                lon = parse_coord(lon_raw)
            except Exception:
                # Skip rows that aren't actual peaks or have weird formatting
                continue

            peak_id, lat5, lon5 = make_peak_id(lat, lon)

            # De-dupe on (name, lat5, lon5)
            key = (peak_name, lat5, lon5)
            if key in seen:
                continue
            seen.add(key)

            rows.append(
                {
                    "peak_id": peak_id,
                    "peak_name": peak_name,
                    "state": "CO",
                    "latitude": lat5,
                    "longitude": lon5,
                    "enter_m": 80,
                    "exit_m": 120,
                    "exit_consec_points": 5,
                }
            )

    if not rows:
        raise RuntimeError("Parsed tables but extracted 0 peaks. Likely column/header mismatch.")

    return rows


def write_csv(rows, out_path):
    fieldnames = [
        "peak_id",
        "peak_name",
        "state",
        "latitude",
        "longitude",
        "enter_m",
        "exit_m",
        "exit_consec_points",
    ]
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


if __name__ == "__main__":
    rows = scrape_colorado_14ers()
    print("Collected:", len(rows))
    print("Sample:", rows[0])
    write_csv(rows, OUT_PATH)
    print("Wrote:", OUT_PATH)